# ⚽ Fotbolls-videoanalys med YOLOv8 – Google Colab

Kör cellerna uppifrån och ner.

**Två viktiga saker för att slippa förlora arbete:**
1. Välj GPU: `Körning → Ändra körningstyp → T4 GPU`.
2. Resultat och nedladdad video sparas på **Google Drive** (steg 3), så de överlever även om Colab kopplar ner.

> ⚠️ Colab kopplar ner efter ~90 min inaktivitet. Håll fliken öppen/aktiv under långa körningar (eller använd Colab Pro för längre sessioner).

## 1. Installera beroenden

In [ ]:
!pip install -q ultralytics opencv-python-headless pyyaml tqdm yt-dlp seaborn scipy

## 2. Hämta projektkoden

Klonar det (publika) repot in i Colab-sessionen.

In [ ]:
import os, sys

if not os.path.isdir('/content/football-video-analysis'):
    !git clone https://github.com/ehoukhe/football-video-analysis.git /content/football-video-analysis
else:
    !cd /content/football-video-analysis && git pull

os.chdir('/content/football-video-analysis')
sys.path.insert(0, os.getcwd())
print('Arbetskatalog:', os.getcwd())

## 3. Anslut Google Drive (så resultat & video överlever omstart)

Första gången öppnas ett fönster där du godkänner åtkomst. Video och resultat sparas sedan under `MyDrive/fotboll/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/fotboll'
DATA_DIR = os.path.join(DRIVE_ROOT, 'data')      # nedladdade videor
OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'output')  # heatmaps, statistik, annoterad video
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Video sparas i :', DATA_DIR)
print('Resultat sparas i:', OUTPUT_DIR)

## 4. Välj video

Ange en YouTube-URL eller ladda upp en fil. Finns videon redan på Drive laddas den **inte** ner igen.

In [ ]:
from pathlib import Path
from src.video import download_youtube

YOUTUBE_URL = ''  # klistra in en URL här, eller lämna tomt för uppladdning

existing = sorted(Path(DATA_DIR).glob('*.mp4'))
if YOUTUBE_URL:
    if existing:
        video_path = str(existing[0])
        print('Använder redan nedladdad video:', video_path)
    else:
        video_path = download_youtube(YOUTUBE_URL, DATA_DIR)
elif existing:
    video_path = str(existing[0])
    print('Använder befintlig video på Drive:', video_path)
else:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    video_path = os.path.join(DATA_DIR, name)
    os.replace(name, video_path)

print('Video:', video_path)

## 5. Kör analysen

`max_frames` styr hur mycket som analyseras. Börja med ett **snabbtest** för att bedöma kvaliteten; sätt `0` för hela matchen när du är nöjd.

Fart-inställningar: `half=True` (FP16 på T4) och `imgsz=640`. Höj `imgsz` till 960/1280 om bollen missas ofta (långsammare).

In [ ]:
from src.pipeline import analyze_video, load_config

cfg = load_config('config/config.yaml')
cfg['model']['device'] = '0'        # GPU
cfg['model']['half'] = True         # FP16 – snabbare på T4
cfg['model']['imgsz'] = 640         # höj vid behov för bättre bolldetektering
cfg['output']['dir'] = OUTPUT_DIR   # spara resultat på Drive
cfg['video']['max_frames'] = 3000   # snabbtest (~2 min film); sätt 0 för hela matchen

result = analyze_video(video_path, cfg)
print('Klar! Resultat sparat i', OUTPUT_DIR)

## 6. Resultat: statistik och insikter

In [ ]:
import json
print(json.dumps(result.stats.summary(), ensure_ascii=False, indent=2))
print('\n=== Insikter till tränaren ===')
for i, ins in enumerate(result.insights, 1):
    print(f'{i}. {ins}')

## 7. Visa heatmaps

In [ ]:
from IPython.display import Image, display
for hm in result.heatmaps:
    display(Image(filename=hm))

## 8. Resultatfilerna

Allt ligger redan kvar på din Google Drive under `MyDrive/fotboll/output/` (annoterad video, heatmaps, `*_stats.json`) – även om Colab kopplar ner. Kör cellen nedan för att lista dem.

In [ ]:
for f in sorted(Path(OUTPUT_DIR).iterdir()):
    print(f.name, f'({f.stat().st_size/1e6:.1f} MB)')